In [1]:
import cv2
import pandas as pd
import supervision as sv
import os
from datetime import datetime
from ultralytics import YOLO
import pymysql
import mysql_account_info as sql_info_JCON
import PySimpleGUI as sg

In [2]:
def db_get_info(table_name,start_date,end_date):
    try:
        #資料庫連線設定
        db = pymysql.connect(host='localhost', port=3306, user='root', passwd=sql_info.password, db='test', charset='utf8')
        cursor=db.cursor()
        sql2="SELECT * FROM "+table_name+" WHERE event_time>='"+str(start_date)+"'AND event_time<='"+str(end_date)+"';"
        df = pd.read_sql(sql2, db)
    except:
        print("Erro:Could not get information from database")
    return df

In [3]:
def db_layout(table_name):
    df=[]
    sg.theme('LightGreen7')
    multi = sg.Multiline('No message', size = (50,20),font = ('宋体',12),key = '-MULTILINE KEY-',disabled=True)
    
    layout = [
              [sg.Text('事件簿',font = ('宋体',12))],
              [multi],
              [sg.Input(key = "-IN-START-",size = 10,font = ('宋体',12),readonly = True)
               ,sg.CalendarButton('開始日期', target="-IN-START-",format='%Y-%m-%d',font = ('宋体',12),size = 10)
               ,sg.Text('',size=6)
               ,sg.Button("查詢",key = "-SEARCH-",font = ('宋体',12),size=6)],
              [sg.Input(key = "-IN-END-",size = 10,font = ('宋体',12),readonly = True)
               ,sg.CalendarButton('結束日期', target="-IN-END-",format='%Y-%m-%d',font = ('宋体',12),size = 10)
               ,sg.Text('',size=6)
               ,sg.Button("下載",key = "-DOWNLOAD-",font = ('宋体',12),size=6)]
              ]
    
    window2 = sg.Window("工廠智慧巡檢系統", layout)
    download_check=False
    
    while True:
        event, values = window2.read()
        if event == sg.WIN_CLOSED:break
        
        if event == "-SEARCH-":  #查詢的東西
            if (values['-IN-START-']=="") or (values['-IN-END-']==""):#沒有開始結束就查詢
                sg.popup_ok('請輸入開始日期與結束日期再進行查詢',title="錯誤")
            else:
                sdate = values['-IN-START-']  #2023-09-14 type=str
                edate = values['-IN-END-']  
                d1 = datetime.strptime(sdate, '%Y-%m-%d') #type=date
                d2 = datetime.strptime(edate, '%Y-%m-%d')
                if (d2-d1).days < 0:
                    sg.popup_ok('請確認開始日期與結束日期正確與否',title="錯誤")
                else:
                    df=db_get_info(table_name,d1,d2)
                    multite = window2['-MULTILINE KEY-']  #這邊應該按下要把東西全部清空 輸入讀進去的東西
                    multite.update(df)
                    download_check=True
        ##################################################
        if event == "-DOWNLOAD-":
            if download_check==True:
                df.to_csv('D:/yolo_csv_test/event_table.csv', sep='\t', encoding='utf-8')
                
        cv2.waitKey(1)

In [4]:
if __name__ == "__main__":
    table_name="check_test"
    db_layout(table_name)

C:\Users\a9603\AppData\Local\Temp\ipykernel_17232\3255709366.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql2, db)
